# GamaX1 (Aetherion) — Colab GPU Training (Bulk / `--data_dir` path)

**Uses the new `bulk_corpus.py` module**, not a single combined `.txt` file. This is the correct path for a large folder of book files (your 2.2GB+ corpus):

- Never loads the whole corpus into RAM as one giant string — it memory-maps an on-disk int32 token cache.
- Prints progress every 500 books while building the cache, so long runs never look frozen.
- Reuses the cache automatically on re-run (checked by file manifest + tokenizer config), so a Colab disconnect doesn't cost you a re-encode.

**Setup order (important):**
1. `Runtime` → `Change runtime type` → Hardware accelerator = **T4 GPU** (or better) → Save.
2. Run cells top to bottom.
3. Put your book `.txt` files in one folder (subfolders OK, they're found recursively) — do NOT pre-combine them into one file. `bulk_corpus.py` handles that internally, book by book.

## 1. Confirm GPU is attached

In [ ]:
!nvidia-smi

If this errors out or shows no GPU, go back to `Runtime` → `Change runtime type`, pick a GPU, then re-run this cell.

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_ROOT = '/content/drive/MyDrive/Aetherion_GamaX1'
BOOKS_DIR    = f'{PROJECT_ROOT}/data/books'                 # folder of .txt book files (2.2GB+)
BULK_CACHE_DIR = f'{PROJECT_ROOT}/bulk_cache_2p2gb'          # memory-mapped token cache lives here
CKPT_DIR     = f'{PROJECT_ROOT}/checkpoints_bulk_2p2gb'      # model checkpoints live here

os.makedirs(BOOKS_DIR, exist_ok=True)
os.makedirs(BULK_CACHE_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
print('Books dir  :', BOOKS_DIR)
print('Bulk cache :', BULK_CACHE_DIR)
print('Checkpoints:', CKPT_DIR)

## 3. Get the GamaX1 codebase onto the runtime

**Important:** make sure the version you clone/copy here actually contains `gamax1/bulk_corpus.py` and the `--data_dir` flag in `train.py` — if you pushed those files to GitHub, a plain `git clone`/`git pull` will pick them up. If they only exist locally and were never pushed, Option A (copy from Drive) is safer until you push.

In [ ]:
# Option A -- code already copied into Drive (edit this path if needed)
CODE_SRC_IN_DRIVE = f'{PROJECT_ROOT}/GamaX1_Aetherion_v1'

%cd /content
!rm -rf gamax1_project
!cp -r "$CODE_SRC_IN_DRIVE" /content/gamax1_project
%cd /content/gamax1_project
!ls gamax1/

In [ ]:
# Option B -- from GitHub instead (uncomment and edit if you pushed there)
# %cd /content
# !rm -rf gamax1_project
# !git clone https://github.com/mrroy-dev/gamax1.git gamax1_project
# %cd /content/gamax1_project
# !ls gamax1/

In [ ]:
# Sanity check: bulk_corpus.py and the --data_dir flag must both be present.
assert os.path.exists('gamax1/bulk_corpus.py'), 'bulk_corpus.py missing -- wrong code version cloned/copied!'
assert '--data_dir' in open('gamax1/train.py').read(), '--data_dir flag missing from train.py -- wrong code version!'
print('Bulk-training code confirmed present.')

## 4. Confirm the books are in place

Upload your `.txt` book files into `BOOKS_DIR` (via the Drive web UI, or `rclone`/`gdown`) beforehand -- as separate files, not pre-merged.

In [ ]:
from pathlib import Path

book_files = sorted(Path(BOOKS_DIR).rglob('*.txt'))
total_bytes = sum(p.stat().st_size for p in book_files)
print(f'Found {len(book_files):,} .txt files, {total_bytes / (1024**3):.2f} GB total')
assert book_files, f'No .txt files found under {BOOKS_DIR} -- upload your books first.'

## 5. Install dependencies

In [ ]:
!pip install -q torch --extra-index-url https://download.pytorch.org/whl/cu121
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only -- check runtime type!')

## 6. Build/reuse the bulk token cache, then train (auto-resume)

First run: this walks every book, encodes it with BPE, and writes the memory-mapped `tokens.int32.bin` cache -- printing progress every 500 books, so you'll see it moving instead of nothing at all. This can take a while for 2.2GB the *first* time; every run after that reuses the cache instantly (unless you pass `--rebuild_bulk_cache`).

Training itself auto-resumes from the newest checkpoint in `CKPT_DIR`, same as before -- safe against disconnects.

**Model size:** left at the 1024/16/10/4096 config sized for a ~1.85B-token corpus (~13-14 tokens/param). Adjust below if your actual book collection ends up smaller or larger once the cache reports its real token count.

In [ ]:
import glob

ckpts = sorted(
    glob.glob(f'{CKPT_DIR}/gamax1_step_*.pt'),
    key=lambda p: int(p.rsplit('_', 1)[1].split('.')[0])
)
resume_flag = f'--resume_from "{ckpts[-1]}"' if ckpts else ''
print('Resuming from:', ckpts[-1] if ckpts else '(no checkpoint found -- starting fresh)')

D_MODEL    = 1024
N_HEADS    = 16
N_LAYERS   = 10
N_FEATURES = 4096

train_cmd = (
    f'python -m gamax1.train '
    f'--tokenizer bpe '
    f'--data_dir "{BOOKS_DIR}" '
    f'--bulk_cache_dir "{BULK_CACHE_DIR}" '
    f'--d_model {D_MODEL} '
    f'--n_heads {N_HEADS} '
    f'--n_layers {N_LAYERS} '
    f'--n_features {N_FEATURES} '
    f'--bpe_vocab_size 8000 '
    f'--max_steps 20000 '
    f'--checkpoint_interval 500 '
    f'--out_dir "{CKPT_DIR}" '
    f'{resume_flag}'
).strip()

print(train_cmd)

In [ ]:
!{train_cmd}

## 7. Generate a sample once training is done (or paused)

In [ ]:
!python -m gamax1.generate --ckpt "{CKPT_DIR}/gamax1.pt" --prompt "Prince Andrew" --max_new_tokens 500

## Notes / gotchas

- **First-run cache build takes real time** for 2.2GB of books -- watch for the `Encoded N/M books | K tokens` progress lines. If you don't see any output for several minutes AND no progress lines, something's actually wrong (wrong code version, or a single giant book file inside `BOOKS_DIR`) -- otherwise, let it run.
- **If disconnected during the cache build itself** (not training): the partial `tokens.int32.bin` is incomplete and will NOT be marked reusable (the metadata file is only written after the full pass completes), so re-running cell 6 will restart the cache build from book 1. This is the one stage that isn't resumable -- budget a single session for it if possible, or pass a smaller subset of `BOOKS_DIR` first to validate everything works before committing to the full 2.2GB.
- **If disconnected during training** (cache already built): re-run cells 1-5, then cell 6 -- the cache is reused instantly and training resumes from the latest checkpoint.
- **`--rebuild_bulk_cache`**: add this flag to `train_cmd` if you ever change the book files or vocab size and need to force a fresh encode.
- Free tier session cap is ~12 hours with no GPU guarantee at peak times; Colab Pro or RunPod/Vast.ai are fallbacks if you hit a wall.